In [ ]:
import pandas as pd
import ollama

In [ ]:
dataset_name= "BanFakeNews"
model_name= "gemma_2B"
prompting = "few_shot"

In [ ]:
dataset_path = "Datasets/BanFakeNews.csv"

In [ ]:
df = pd.read_csv(dataset_path)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
messages = [
    {
        'role': 'system',
        'content': """You are an expert in news classification. Classify the sentiment of the following texts into one of two categories: 
            0: fake
            1: authentic

        Examples:
        Text: "বইটি পড়ে খুব খারাপ লাগলো, সময় নষ্ট।"
        Label: 0

        Text: "বইটি আমার খুব ভালো লেগেছে।"
        Label: 1

        MUST REMEMBER: DO NOT PUT ANY OTHER STRING. ONLY ONE LABEL EITHER 0 or 1. 

        Now provide label the following text:
        """,
    }
]


In [ ]:
def main(column_to_annotate):
    max_history_length = 1500  
    
    output_file = f"annotated_results_{dataset_name}_{model_name}_{prompting}.csv"
    
    with open(output_file, "w") as output_file:
        output_file.write("Input,Label\n")  

        for line in df[column_to_annotate]:
            # Append the new input to messages
            messages.append({'role': 'user', 'content': f'Input: {line}'})
 
            # Build messages to send, keeping the last N messages
            messages_to_send = [messages[0]] + messages[-(max_history_length - 1):]
 
            # Send the messages to the model
            response = ollama.chat(model='gemma2:2b', messages=messages_to_send)
 
            # Get the model's response
            output = response['message']['content'].strip()
            
            # print(output)
 
            # Append the model's response to messages
            messages.append({'role': 'assistant', 'content': output})
 
            # Extract the sentiment label from the model's response
            label = output.replace("Label: ", "").strip()
            
            # print(label)
 
            # Write the input and bucket ID to the CSV file
            output_file.write(f'"{line}","{label}"\n')
 
            # Optional: print progress
            print(f">>> {line}\n{label}\n")
    
    print("Annotation Has been done!!")

In [ ]:
main(column_to_annotate="Data")


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Load the CSV files
llm_csv_path = f"annotated_results_{dataset_name}_{model_name}_{prompting}.csv"
human_csv_path = dataset_path 

# Read CSV files into dataframes
llm_df = pd.read_csv(llm_csv_path)
human_df = pd.read_csv(human_csv_path)

# Extract labels
llm_labels = llm_df['Label']
human_labels = human_df['Label']

# Calculate metrics
accuracy = accuracy_score(human_labels, llm_labels)
precision = precision_score(human_labels, llm_labels, average='weighted')
recall = recall_score(human_labels, llm_labels, average='weighted')
f1 = f1_score(human_labels, llm_labels, average='weighted')
conf_matrix = confusion_matrix(human_labels, llm_labels)
report = classification_report(human_labels, llm_labels)

# Display metrics
print("Metrics Analysis:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(report)

# Optional: Save results to a file
output_path = f"metrics_analysis_{dataset_name}_{model_name}_{prompting}.txt"
with open(output_path, "w") as output_file:
    output_file.write(f"Metrics Analysis:\n")
    output_file.write(f"Accuracy: {accuracy:.2f}\n")
    output_file.write(f"Precision: {precision:.2f}\n")
    output_file.write(f"Recall: {recall:.2f}\n")
    output_file.write(f"F1 Score: {f1:.2f}\n")
    output_file.write("\nConfusion Matrix:\n")
    output_file.write(f"{conf_matrix}\n")
    output_file.write("\nClassification Report:\n")
    output_file.write(f"{report}\n")
